In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


# **Ensemble Learning**

Ensemble Learning é o nome dado para a técnica de se utilizar varios preditores juntos em um grupo chamado ensemble em que trabalhem juntos de alguma forma/metodo (ensemble method) para melhorar a acurácia. Toda técnica de ensemble tem premissa de independencia entre individuos do grupo pois a ideia é que eles aprendam coisas diferentes para se complementarem, ou seja, um mitigar o erro do outro.

## **Voting Ensemble Method** 

Essa técnica de ensemble consiste em treinar um grupo de modelos independentes, ou seja, dados diferentes, algoritmos de treino diferentes ou hyperparametros diferentes. Na inferencia usamos o sistema de votos para entre os modelos para gerar o resultado, existem 2 formas de fazer isso:

- **Hard Voting**

Cada modelo faz uma predicao escolhendo uma classe e pegamos a classe que mais aparece entre o grupo de classificadores

- **Soft Voting**

Ao invés da classe, pegamos as *probabilidades das classes de todos*, somamos e tiramos a media de cada classe, pegamos a *classe com maior media de probabilidade* (Para usar soft todos os modelos tem que conseguir prever probabilidade). Isso é util pois permite que predicoes com *maior confianca recebam mais peso* pois a media será maior e além disso permite aplicacao em regressao em que podemos calcular a media das predicoes.


In [3]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42)

voting_clf = VotingClassifier(estimators=[
    ("lr", LogisticRegression()),
    ("rf", RandomForestClassifier()),
    ("svc", SVC())], voting="hard")

voting_clf.fit(X_train, y_train)

for clf in (LogisticRegression(), RandomForestClassifier(), SVC()):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, clf.score(X_test, y_test))

print("Voting Classifier", voting_clf.score(X_test, y_test))


    

LogisticRegression 0.864
RandomForestClassifier 0.888
SVC 0.896
Voting Classifier 0.904


## **Bagging e Pasting**

Uma forma outra forma de fazer um ensemble é treinar o mesmo algoritmo usando diferentes amostragens dos dados, isso pode ser feito de duas formas:

- **Bagging:** Cada modelo é treinado com uma amostra de mesmo tamanho do original gerada usando bootstrap nos dados originais, ou seja, usando amostragem com reposicao.
- **Pasting:** Mesma logica do bagging porém com amostragem sem reposicao.

Uma vez treinados a inferencia é feita de forma parecida com o hard voting ou soft voting. Uma outra vantagem de ensembles é que geralmente os preditores podem ser *treinados em paralelo e a inferencia também* pode ser feita em paralelo e isso faz eles escalarem muito bem.

Ensemble tende a *reduzir a variancia* dos modelos enquanto mantem o *vies bem estavel*, isso acontece pois a variancia, que é o erro aleatorio nas predicoes que o modelo aprendeu no treino devido a sua sensibilidade alta a variacoes, se cancela com o erro de outros modelos treinados com algoritmos/dados diferentes. O bias por outro lado, que é a tendencia/capacidade do algortimo para aprender padroes continua constante. 

- Por esse motivo existe ensemble de arvores que tem varianca alta pois tem capacidade/sensibilidade de overfittar facilmente e vies baixo pois conseguem aprender padroes complexos e nao lineares. 
- Por outro lado nao existe de regressoes lineares que tem alto vies devido a assumir linearidade e baixa variancia pois nao tem tanta capacidade de se adaptar ao ruido.

Bagging pode reduzir a variancia do modelo pois o *bootstrap pode ajudar a cancelar os erros* enquanto o pasting pode ser mais *barato computacionalmente e funciona bem se a variancia ja for baixa*.

### **Out-of-bag evaluation**

Durante o bagging, a amostragem do bootstrap para cada preditor costuma deixar cerca de 37% das instancias originais sem nunca ser amostrada para o preditor. Esse grupo de amostras que o preditor nunca viu sao chamadas de dados out-of-bag e podem ser usados como uma estimativa de como o ensemble vai se comportar com dados nunca vistos. Isso é chamado de out-of-bag evaluation.

### **Random Patches e Random Subspaces**

Bagging suporta também bootstrap de features ao invés de apenas dos dados e isso é particularmente util ao lidar com problemas de alta cardinalidade.

- **Random Patches:** Usar bootstrap tanto nas features como nas linhas
- **Random Subspaces:** Usar bootstrap apenas nas features


## **Random Forest**


In [7]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=1000,max_samples=100, n_jobs=-1, random_state=42)

bag_clf.fit(X_train, y_train)

print("Bagging Classifier", bag_clf.score(X_test, y_test))

Bagging Classifier 0.904
